In [1]:
# Run this first to train and save the model
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("failure_risk_dataset.csv")

# Remove leakage
df = df.drop(columns=["passed_tests"])

if "test_error" in df.columns:
    df = df.drop(columns=["test_error"])

target = "failed"
df = df.dropna(subset=[target])

# =========================
# 2. FEATURES
# =========================
X = df.drop(columns=[target])
y = df[target]

# Encode target
if y.dtype == "object":
    le = LabelEncoder()
    y = le.fit_transform(y)
    # Save label encoder for later use
    joblib.dump(le, "label_encoder.pkl")

# Feature engineering (important)
X["complexity_per_len"] = X["avg_complexity"] / (X["code_len"] + 1)
X["ast_per_len"] = X["ast_nodes"] / (X["code_len"] + 1)
X["log_code_len"] = np.log1p(X["code_len"])

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Fill missing
X = X.fillna(X.median())

# =========================
# 3. SPLIT
# =========================
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

# =========================
# 4. SCALING
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, "scaler.pkl")

# Save feature names for later inference
feature_names = X.columns.tolist()
joblib.dump(feature_names, "feature_names.pkl")

# =========================
# 5. TENSORS
# =========================
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(np.array(y_train), dtype=torch.float32).view(-1,1)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(np.array(y_val), dtype=torch.float32).view(-1,1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(np.array(y_test), dtype=torch.float32).view(-1,1)

# =========================
# 6. DATALOADER
# =========================
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=32,
    shuffle=True
)

# =========================
# 7. MODEL
# =========================
class ANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.4),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)  # NO sigmoid
        )

    def forward(self, x):
        return self.net(x)

model = ANN(X_train.shape[1])

# =========================
# 8. LOSS + OPTIMIZER
# =========================
pos_weight = torch.tensor([(len(y_train) - sum(y_train)) / sum(y_train)])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# =========================
# 9. TRAINING
# =========================
best_val_loss = float('inf')
patience = 7
counter = 0
best_model_state = None

# Create models directory if it doesn't exist
os.makedirs("models", exist_ok=True)

for epoch in range(100):
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_t)
        val_loss = criterion(val_outputs, y_val_t)

    print(f"Epoch {epoch}, Val Loss: {val_loss.item():.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model_state = model.state_dict().copy()
        # Save checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, "models/best_model_checkpoint.pt")
        print(f"  -> New best model saved at epoch {epoch}")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

# Load the best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Loaded best model based on validation loss")

# =========================
# 10. EVALUATION
# =========================
model.eval()
with torch.no_grad():
    y_test_logits = model(X_test_t)
    y_test_prob = torch.sigmoid(y_test_logits).numpy()

# =========================
# 11. THRESHOLD TUNING
# =========================
best_acc = 0
best_t = 0.5

for t in np.arange(0.3, 0.8, 0.05):
    preds = (y_test_prob > t).astype(int)
    acc = accuracy_score(y_test, preds)

    if acc > best_acc:
        best_acc = acc
        best_t = t

print("\nBest Threshold:", best_t)
print("Best Accuracy:", best_acc)

# Final predictions
y_pred = (y_test_prob > best_t).astype(int)

# =========================
# 12. FINAL METRICS
# =========================
print("\n===== FINAL METRICS =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_test_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# =========================
# 13. SAVE FINAL MODEL
# =========================
# Save the final model with all metadata
final_model_path = "models/failure_risk_model.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': X_train.shape[1],
    'best_threshold': best_t,
    'model_architecture': {
        'input_dim': X_train.shape[1],
        'hidden_layers': [256, 128, 64],
        'output_dim': 1
    },
    'metrics': {
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_test_prob),
        'best_threshold': best_t
    }
}, final_model_path)

print(f"\n✅ Model saved to {final_model_path}")

# Save the best threshold for inference
with open("models/best_threshold.txt", "w") as f:
    f.write(str(best_t))

print("\n✅ All artifacts saved:")
print("   - models/best_model_checkpoint.pt")
print("   - models/failure_risk_model.pth")
print("   - models/best_threshold.txt")
print("   - scaler.pkl")
print("   - feature_names.pkl")

Epoch 0, Val Loss: 0.4060
  -> New best model saved at epoch 0
Epoch 1, Val Loss: 0.3839
  -> New best model saved at epoch 1
Epoch 2, Val Loss: 0.3781
  -> New best model saved at epoch 2
Epoch 3, Val Loss: 0.3743
  -> New best model saved at epoch 3
Epoch 4, Val Loss: 0.3717
  -> New best model saved at epoch 4
Epoch 5, Val Loss: 0.3733
Epoch 6, Val Loss: 0.3688
  -> New best model saved at epoch 6
Epoch 7, Val Loss: 0.3699
Epoch 8, Val Loss: 0.3677
  -> New best model saved at epoch 8
Epoch 9, Val Loss: 0.3650
  -> New best model saved at epoch 9
Epoch 10, Val Loss: 0.3674
Epoch 11, Val Loss: 0.3633
  -> New best model saved at epoch 11
Epoch 12, Val Loss: 0.3627
  -> New best model saved at epoch 12
Epoch 13, Val Loss: 0.3639
Epoch 14, Val Loss: 0.3649
Epoch 15, Val Loss: 0.3678
Epoch 16, Val Loss: 0.3706
Epoch 17, Val Loss: 0.3697
Epoch 18, Val Loss: 0.3691
Epoch 19, Val Loss: 0.3693
Early stopping triggered
Loaded best model based on validation loss

Best Threshold: 0.3
Best Accu